# ShadowMesh — Anomaly Detector & RL Topology Optimizer

This notebook demonstrates both production AI models from the ShadowMesh cyber deception platform:

1. **IsolationForest Anomaly Detector** — scores attacker actions against a benign baseline
2. **Q-Learning Topology Optimizer** — selects which deception network layout maximizes attacker engagement

Both models were trained on real Kali Linux attack data captured from live Docker honeypots.

---
**Companion dataset:** [ShadowMesh Cyber Deception Dataset](https://www.kaggle.com/datasets/deaker09/shadowmesh-cyber-deception-dataset)  
**Source code:** [github.com/Deaker-09/ShadowMess-AI](https://github.com/Deaker-09/ShadowMess-AI)

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#0d0d0d'
plt.rcParams['axes.facecolor'] = '#1a1a1a'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['grid.color'] = '#333333'

SHADOW_RED    = '#E24B4A'
SHADOW_GREEN  = '#1D9E75'
SHADOW_AMBER  = '#EF9F27'
SHADOW_PURPLE = '#7F77DD'
SHADOW_BLUE   = '#378ADD'

FEATURES = [
    'timing_delta_ms', 'port_entropy', 'unique_ports_5s',
    'login_rate_60s', 'command_diversity', 'lateral_spread', 'action_type_encoded'
]

TOPOLOGY_NAMES = {
    0: 'hub-and-spoke',
    1: 'mesh',
    2: 'layered',
    3: 'star',
    4: 'random',
    5: 'financial-heavy',
    6: 'dev-heavy',
    7: 'admin-heavy',
}

print('Setup complete')

In [ ]:
import os, glob

def find_input(filename):
    """Finds a file anywhere under /kaggle/input regardless of dataset slug."""
    matches = glob.glob(f'/kaggle/input/**/{filename}', recursive=True)
    if matches:
        return matches[0]
    # Local fallback for running outside Kaggle
    local = os.path.join(os.getcwd(), filename)
    if os.path.exists(local):
        return local
    raise FileNotFoundError(f'{filename} not found in /kaggle/input or local directory')

DATASET_CSV   = find_input('real_attack_dataset.csv')
QTABLE_CSV    = find_input('q_table_readable.csv')
MODEL_PKL     = find_input('isolation_forest_model.pkl')
QTABLE_NPY    = find_input('q_table.npy')

print(f'Dataset CSV : {DATASET_CSV}')
print(f'Q-table CSV : {QTABLE_CSV}')
print(f'Model PKL   : {MODEL_PKL}')
print(f'Q-table NPY : {QTABLE_NPY}')


## Part 1 — IsolationForest Anomaly Detector

### Architecture
- **Algorithm:** IsolationForest (`n_estimators=100`, `contamination=0.05`, `seed=42`)
- **Input:** 7-feature behavioral vector extracted from each attacker action
- **Output:** `threat_score ∈ [0, 1]` — higher = more anomalous
- **Training:** 3,000 benign baseline samples
- **Validation:** 45 real attack vectors from nmap, hydra, sqlmap, OWASP payloads

In [ ]:
model = joblib.load(MODEL_PKL)
print(f'IsolationForest loaded')
print(f'  n_estimators  : {model.n_estimators}')
print(f'  contamination : {model.contamination}')
print(f'  n_features_in : {model.n_features_in_}')
print(f'  feature names : {FEATURES}')

### Score Real Captured Attacks

In [ ]:
df = pd.read_csv(DATASET_CSV)

X = df[FEATURES].values.astype(np.float32)
y = df['label'].values

raw_scores = model.decision_function(X)
threat_scores = np.clip(1.0 - (raw_scores + 0.5), 0.0, 1.0)
predictions = model.predict(X)

benign_acc = (predictions[y == 0] ==  1).mean()
attack_rec = (predictions[y == 1] == -1).mean()

print(f'Performance on real data:')
print(f'  Benign accuracy : {benign_acc:.1%}')
print(f'  Attack recall   : {attack_rec:.1%}')

print(f'\nMean threat scores by phase:')
df['threat_score'] = threat_scores
phase_scores = df.groupby('attack_phase')['threat_score'].mean().sort_values(ascending=False)
for phase, score in phase_scores.items():
    bar = '#' * int(score * 30)
    flag = ' <- ANOMALOUS' if score > 0.5 else ''
    print(f'  {phase:<25} {score:.3f}  {bar}{flag}')

### Usage: Score Any Action

In [ ]:
def score_action(timing_ms, port_entropy, unique_ports_5s, 
                 login_rate, cmd_diversity, lateral_spread, action_type):
    """
    Score a single attacker action.
    
    action_type: port_scan=0, login=1, cmd=2, data=3, lateral=4, cred=5, canary=6
    Returns: threat_score in [0,1] — higher = more anomalous
    """
    features = np.array([[timing_ms, port_entropy, unique_ports_5s,
                          login_rate, cmd_diversity, lateral_spread, action_type]], 
                        dtype=np.float32)
    raw = model.decision_function(features)[0]
    threat_score = float(np.clip(1.0 - (raw + 0.5), 0.0, 1.0))
    is_anomalous = model.predict(features)[0] == -1
    return threat_score, is_anomalous


scenarios = [
    ('Normal scheduled job',        score_action(3000, 0.15, 1,  0, 2, 1, 2)),
    ('nmap port scan (real)',        score_action(  45, 3.80, 22, 0, 1, 4, 0)),
    ('hydra SSH brute force',        score_action( 120, 0.10,  1, 18, 1, 1, 1)),
    ('OWASP SQLi probe',             score_action( 300, 0.00,  0,  0, 3, 1, 2)),
    ('Lateral movement + creds',     score_action( 200, 0.90,  5,  2, 5, 8, 4)),
    ('Ransomware deployment burst',  score_action(  15, 2.50, 12,  0, 6, 5, 2)),
    ('Credential exfiltration',      score_action( 500, 0.30,  2,  1, 4, 3, 5)),
    ('SSRF cloud metadata probe',    score_action( 800, 0.00,  0,  0, 3, 1, 3)),
]

print(f'{"Scenario":<35} {"Threat Score":>12}  {"Verdict"}')
print('-' * 65)
for name, (ts, anom) in scenarios:
    verdict = 'ANOMALOUS' if anom else 'normal'
    bar = '#' * int(ts * 20)
    color_hint = '!!!' if anom else '   '
    print(f'{name:<35} {ts:>12.3f}  {color_hint} {verdict}')

### IsolationForest Decision Boundary Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('IsolationForest Decision Boundary — Key Feature Pairs', 
             fontsize=14, fontweight='bold', y=1.02)

pairs = [
    ('timing_delta_ms', 'port_entropy', 'Timing vs Port Entropy'),
    ('login_rate_60s', 'lateral_spread', 'Login Rate vs Lateral Spread'),
    ('port_entropy', 'command_diversity', 'Port Entropy vs Command Diversity'),
]

benign = df[df['label'] == 0]
attack = df[df['label'] == 1]

for ax, (fx, fy, title) in zip(axes, pairs):
    ax.scatter(benign[fx], benign[fy], 
               c=SHADOW_GREEN, alpha=0.15, s=8, label='Benign')
    ax.scatter(attack[fx], attack[fy], 
               c=SHADOW_RED, alpha=0.9, s=60, zorder=5,
               edgecolors='white', linewidths=0.5, label='Attack')

    # Grid of threat scores
    xi = np.percentile(df[fx], [1, 99])
    yi = np.percentile(df[fy], [1, 99])
    xx, yy = np.meshgrid(np.linspace(*xi, 60), np.linspace(*yi, 60))
    
    fi = FEATURES.index(fx)
    gi = FEATURES.index(fy)
    grid_input = np.zeros((xx.ravel().shape[0], 7), dtype=np.float32)
    grid_input[:, fi] = xx.ravel()
    grid_input[:, gi] = yy.ravel()
    grid_input[:, 0] = benign['timing_delta_ms'].mean()

    raw = model.decision_function(grid_input)
    ts = np.clip(1.0 - (raw + 0.5), 0, 1).reshape(xx.shape)

    contour = ax.contourf(xx, yy, ts, levels=20, cmap='RdYlGn_r', alpha=0.25)
    ax.contour(xx, yy, ts, levels=[0.5], colors=[SHADOW_AMBER], linewidths=1.5, linestyles='--')

    ax.set_xlabel(fx, fontsize=10)
    ax.set_ylabel(fy, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('decision_boundaries.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

---
## Part 2 — Q-Learning Topology Optimizer

### Architecture
- **Algorithm:** Tabular Q-learning
- **State space:** 2,880 bins (6 variables discretized)
- **Action space:** 8 deception topology configurations
- **Reward:** engagement quality — unique MITRE techniques (+2), credential access (+3), dwell time bonus
- **Training:** 50,000 simulated episodes, reward shaped by real attack phase distribution

The RL agent answers: *"Given this attacker's current profile, which fake network layout will make them spend the most time — and reveal the most intelligence?"*

In [ ]:
q_table = np.load(QTABLE_NPY)
print(f'Q-table loaded: {q_table.shape} (states x actions)')
print(f'Non-zero states: {np.count_nonzero(q_table.sum(axis=1)):,} / {q_table.shape[0]:,}')
print(f'Max Q-value: {q_table.max():.2f}')
print(f'Min Q-value: {q_table.min():.2f}')

best_actions = np.argmax(q_table, axis=1)
from collections import Counter
action_dist = Counter(best_actions)
print(f'\nLearned topology preferences across all states:')
for action, count in sorted(action_dist.items(), key=lambda x: -x[1]):
    pct = count / len(best_actions) * 100
    bar = '#' * int(pct / 2)
    print(f'  {TOPOLOGY_NAMES[action]:<20} {count:>5} states ({pct:>5.1f}%)  {bar}')

### Topology Selection by Attacker Profile

In [ ]:
BIN_EDGES = {
    'actions':      [0, 5, 15, 30, 50],
    'generation':   [0, 3, 7],
    'duration':     [0, 5, 20, 60],
    'nodes_hit':    [0, 3, 8, 15],
    'cred_attempts':[0, 1, 5],
}

SKILL_MAP = {'script_kiddie': 0, 'intermediate': 1, 'advanced': 2, 'nation_state_apt': 3}

def digitize(v, edges):
    for i, e in enumerate(edges):
        if v <= e: return i
    return len(edges)

def state_index(skill, actions, generation, duration_min, nodes_hit, cred_attempts):
    s = [
        min(skill, 3),
        digitize(actions,       BIN_EDGES['actions']),
        digitize(generation,    BIN_EDGES['generation']),
        digitize(duration_min,  BIN_EDGES['duration']),
        digitize(nodes_hit,     BIN_EDGES['nodes_hit']),
        digitize(cred_attempts, BIN_EDGES['cred_attempts']),
    ]
    idx = s[0]*(5*3*4*4*3) + s[1]*(3*4*4*3) + s[2]*(4*4*3) + s[3]*(4*3) + s[4]*3 + s[5]
    return min(idx, 2879)

def recommend_topology(skill_label, actions, generation, duration_min, nodes_hit, cred_attempts):
    skill = SKILL_MAP.get(skill_label, 0)
    state = state_index(skill, actions, generation, duration_min, nodes_hit, cred_attempts)
    best = int(np.argmax(q_table[state]))
    q_vals = q_table[state]
    return TOPOLOGY_NAMES[best], best, q_vals


print(f'{"Attacker Profile":<40} {"Recommended Topology":<22} {"Confidence"}')
print('-' * 80)

profiles = [
    ('Script Kiddie — early stage',           'script_kiddie',    3,  0,  3.0,  2,  0),
    ('Intermediate — active recon',           'intermediate',    12,  1,  8.0,  5,  1),
    ('Advanced — credential hunting',         'advanced',        25,  2, 25.0,  9,  3),
    ('Advanced — long dwell, many nodes',     'advanced',        40,  3, 60.0, 14,  5),
    ('Nation-State APT — early',              'nation_state_apt', 8,  1, 10.0,  4,  2),
    ('Nation-State APT — late stage',         'nation_state_apt',50,  5, 80.0, 18,  8),
    ('Nation-State APT — financial target',   'nation_state_apt',35,  4, 45.0, 12,  6),
]

for desc, skill, actions, gen, dur, nodes, creds in profiles:
    topo, action_id, q_vals = recommend_topology(skill, actions, gen, dur, nodes, creds)
    confidence = float(q_vals.max() - q_vals.min())
    print(f'{desc:<40} {topo:<22} Q-spread={confidence:.2f}')

### Q-Table Heatmap — Learned Policy

In [ ]:
q_readable = pd.read_csv(QTABLE_CSV)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Q-Learning Topology Optimizer — Learned Policy', fontsize=14, fontweight='bold', y=1.01)

# Left: topology preference distribution
ax = axes[0]
topo_counts = q_readable['topology_name'].value_counts()
colors = [SHADOW_RED, SHADOW_AMBER, SHADOW_GREEN, SHADOW_PURPLE, 
          SHADOW_BLUE, '#D4537E', '#C97B2A', '#2AB5C9']
bars = ax.barh(topo_counts.index, topo_counts.values, 
               color=colors[:len(topo_counts)], alpha=0.85, edgecolor='none')
ax.set_xlabel('Number of states preferring this topology', fontsize=11)
ax.set_title('Topology Preference Distribution\n(across 2,880 attacker states)', fontsize=11, fontweight='bold')
for bar, val in zip(bars, topo_counts.values):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2, 
            f'{val} ({val/2880*100:.1f}%)', va='center', fontsize=9, color='white')
ax.grid(True, axis='x', alpha=0.3)

# Right: Q-value spread (confidence) per topology
ax2 = axes[1]
q_cols = [f'q_action_{i}' for i in range(8)]
q_matrix = q_readable[q_cols].values

# Normalize for display
q_min, q_max = q_matrix.min(), q_matrix.max()
q_norm = (q_matrix - q_min) / (q_max - q_min + 1e-9)

# Sample 100 states for readability
sample_idx = np.linspace(0, len(q_norm)-1, 100, dtype=int)
im = ax2.imshow(q_norm[sample_idx].T, aspect='auto', cmap='inferno', 
                interpolation='nearest')
ax2.set_yticks(range(8))
ax2.set_yticklabels([TOPOLOGY_NAMES[i] for i in range(8)], fontsize=9)
ax2.set_xlabel('State (sampled)', fontsize=11)
ax2.set_title('Q-Values Heatmap (100 sampled states)\nBrighter = higher Q-value = preferred', 
              fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax2, shrink=0.8, label='Normalized Q-value')

plt.tight_layout()
plt.savefig('qtable_policy.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

### How Topology Choice Changes with Attacker Skill

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('RL Topology Recommendations by Attacker Skill Level\n(as dwell time increases)', 
             fontsize=14, fontweight='bold', y=1.01)

skill_configs = [
    (0, 'Script Kiddie',     SHADOW_GREEN),
    (1, 'Intermediate',      SHADOW_AMBER),
    (2, 'Advanced',          SHADOW_RED),
    (3, 'Nation-State APT',  SHADOW_PURPLE),
]

durations = np.linspace(0, 90, 50)

for ax, (skill_id, skill_name, color) in zip(axes.flatten(), skill_configs):
    recommended = []
    q_spreads = []

    for dur in durations:
        actions = max(1, int(dur * 0.8))
        nodes   = max(1, int(dur * 0.15))
        creds   = max(0, int(dur * 0.05))
        gen     = max(0, int(dur * 0.03))
        state   = state_index(skill_id, actions, gen, dur, nodes, creds)
        best    = int(np.argmax(q_table[state]))
        spread  = float(q_table[state].max() - q_table[state].min())
        recommended.append(best)
        q_spreads.append(spread)

    topo_colors = plt.cm.tab10(np.array(recommended) / 8)
    for i in range(len(durations) - 1):
        ax.fill_between(
            durations[i:i+2], recommended[i:i+2],
            alpha=0.8, color=plt.cm.tab10(recommended[i] / 8)
        )

    ax.step(durations, recommended, color='white', linewidth=1.5, where='post')
    ax.set_yticks(range(8))
    ax.set_yticklabels([TOPOLOGY_NAMES[i] for i in range(8)], fontsize=8)
    ax.set_xlabel('Dwell Time (minutes)', fontsize=10)
    ax.set_title(f'{skill_name}', fontsize=12, fontweight='bold', color=color)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('topology_by_skill.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print('Nation-State APTs trigger more specialized topologies (financial-heavy, admin-heavy)')
print('Script Kiddies get simpler hub-and-spoke — less resource intensive')

## Summary

| Model | What It Does | Performance |
|---|---|---|
| IsolationForest | Scores each attacker action 0–1 for anomaly | 95% benign acc, 100% attack recall |
| Q-table | Selects deception topology per attacker profile | 2,880 states, 8 topology configs |

Both models run in the ShadowMesh production backend — the IsolationForest scores every incoming action in real time, while the Q-table optimizer is consulted each time a topology mutation is triggered.

**Source code:** [github.com/Deaker-09/ShadowMess-AI](https://github.com/Deaker-09/ShadowMess-AI)